# Ocean sounds: generators trained on scarce data

For the scarcity experiment (`ocean_scarcity.ipynb`). The existing ocean Path A model was trained
on all ~25,000 training clips, so its samples carry information a data-scarce classifier would
not have. To test honestly whether generating more data helps when recordings are scarce, the
generator must see **only the clips the classifier gets**.

This notebook trains Path A again on **64** and on **256** real clips per class, then samples 500
clips per class from each, and checks each model for memorisation.

| Choice | Reason |
|---|---|
| The same model and recipe as ocean Path A (UNet, 128 mel bands, v-prediction, cosine schedule, EMA, 10,000 steps, batch 32, class-balanced draws) | Only the amount of training data changes, so differences from the full-data generator can be attributed to data scarcity. 10,000 steps is about 140 passes over 2,304 clips and about 830 over 384: memorisation is a real risk, which is why it is measured below, not assumed away. |
| The budget's clips are `tstr.scarce_indices(..., seed=0)`: nested (the 64-clip set is inside the 256-clip set) | The scarcity notebook gives the classifier exactly these clips, so the generator never sees a real clip the classifier does not. |
| The mel front end's dB window is fitted on the budget's clips only | Even that one statistic would otherwise come from all the data. |
| A fresh random crop per draw, as in every training run | The 2 s clips are slightly longer than a crop, so this adds a little variation, as for the full-data model. |

**Memorisation check.** For each generated clip, the distance (generator mel space) to its nearest
clip in the budget's training set, against the same distance for unseen validation clips. A ratio
well below 1, or many generated clips closer than 99% of validation clips are, means the model is
reproducing its training clips. That would make "synthetic data" a disguised copy of the real set,
and the scarcity notebook would need to report it that way.

**Resumable.** Training resumes from `last.pt` (saved every 1,000 steps); sampling saves per class.
Runtime on a T4: about 3.5 hours of training per budget, plus 30-60 minutes of sampling. Run one
budget per session if needed: set `BUDGETS`.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, time, subprocess
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C
import tstr as T

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

assert hasattr(T, "scarce_indices"), "old code: pull the repo, then restart the session"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()

## 2. Config

In [ ]:
QUICK_TEST = False          # True: tiny run end to end, numbers meaningless

# data: must match notebooks/ocean_path_a.ipynb
SOURCE      = "/content/drive/MyDrive/thesis_runs/data/ocean_sounds_thesis.zip"
GROUP_SIZE  = 20
HIGHPASS_HZ = 20.0
NORMALIZE   = "rms"
OUT_ROOT    = "/content/drive/MyDrive/thesis_runs/ocean"

BUDGETS     = (64, 256)     # real clips per class each generator is trained on; run one at a time if needed
SUBSET_SEED = 0             # must match ocean_scarcity.ipynb
N_GEN       = 500           # generated clips per class
GUIDANCE    = 1.0
GEN_BATCH   = 64
GL_ITERS    = 64
DROPOUT     = 0.1

overrides = dict(total_steps=300, inference_steps=10) if QUICK_TEST else {}
cfg = C.ocean_config(out_root=OUT_ROOT, data_root=f"{OUT_ROOT}/data", **overrides)
if QUICK_TEST:
    N_GEN, GL_ITERS = 4, 8
SCARCE_ROOT = Path(f"{cfg.out_root}/scarce" + ("_quick" if QUICK_TEST else ""))
def budget_dir(n):
    return SCARCE_ROOT / f"path_a128_hp{HIGHPASS_HZ:g}_{NORMALIZE}_bal_n{n}"
C.seed_everything(cfg.seed)
for n in BUDGETS:
    print(n, "->", budget_dir(n))

## 3. Data

In [ ]:
data = C.load_ocean(cfg, SOURCE, group_size=GROUP_SIZE, highpass_hz=HIGHPASS_HZ, normalize=NORMALIZE)
class_names = data["class_names"]; n_classes = len(class_names)
train_split, val_split, _ = C.split_by_fold(data, cfg)
del data

def center_crop(w):
    off = (w.shape[-1] - cfg.clip_samples) // 2
    return w[..., off:off + cfg.clip_samples]

n_ref = min(200, int(np.bincount(val_split["labels"].numpy()).min()))
val_ref = T.subset_split(val_split, T.scarce_indices(val_split["labels"].numpy(), n_ref, seed=1))   # fixed memorisation reference
print(f"train {len(train_split['labels'])} clips | validation reference for memorisation {len(val_ref['labels'])} clips")

## 4. Train, sample, check: one budget at a time

In [ ]:
fwd = lambda m, x, t, y: m(x, t, class_labels=y).sample
summary = {}
for n in BUDGETS:
    d = budget_dir(n); d.mkdir(parents=True, exist_ok=True)
    idx = T.scarce_indices(train_split["labels"].numpy(), n, seed=SUBSET_SEED)
    sub = T.subset_split(train_split, idx)
    np.save(d / "train_indices.npy", idx)
    print(f"\n==================== {n} clips per class ({len(idx)} clips) ====================")

    fe_file = d / "frontend.json"
    fe = (C.MelFrontEnd.load(fe_file, cfg, device=device) if fe_file.exists()
          else C.MelFrontEnd(cfg, device=device).fit(sub["waveforms"].float()))
    fe.save(fe_file)
    print(f"dB window (fitted on the {len(idx)} clips): [{fe.db_max - cfg.db_range:.1f}, {fe.db_max:.1f}]")

    # train (resumes from last.pt)
    unet = C.build_unet2d(cfg, dropout=DROPOUT).to(device)
    ds = C.BalancedDataset(C.CropDataset(sub, cfg), sub["labels"])
    t0 = time.time()
    ema, losses = C.train_diffusion(unet, fwd, fe, ds, cfg, d / "run", device)
    print(f"training: {(time.time() - t0) / 60:.1f} min this session")
    ema.copy_to(unet.parameters()); unet.eval()

    # sample per class (resumable)
    parts = d / "gen_parts"; parts.mkdir(exist_ok=True)
    for c in range(n_classes):
        f = parts / f"class{c}.pt"
        if f.exists():
            continue
        C.seed_everything(2000 + 10 * n + c)
        x = C.sample(unet, fwd, (1, cfg.n_mels, cfg.spec_frames), cfg, device,
                     labels=torch.full((N_GEN,), c, dtype=torch.long), guidance=GUIDANCE, batch_size=GEN_BATCH)
        torch.save({"samples": x.half(), "label": c}, f)
        print(f"   sampled {class_names[c]}")
    gen_mel = torch.cat([torch.load(parts / f"class{c}.pt")["samples"].float() for c in range(n_classes)]).clamp(-1, 1)
    gen_y = torch.arange(n_classes).repeat_interleave(N_GEN)
    wav_file = d / f"gen_wave_gl{GL_ITERS}.pt"
    if not wav_file.exists():
        torch.save({"waves": C.spec_to_wave(gen_mel, fe, n_iter=GL_ITERS, seed=cfg.seed), "labels": gen_y,
                    "meta": dict(budget=n, n_gen=N_GEN, guidance=GUIDANCE, train_steps=cfg.total_steps, commit=commit)}, wav_file)

    # memorisation: nearest training clip in mel space
    bank = torch.cat([fe(center_crop(sub["waveforms"][i:i + 256]).to(device).float()) for i in range(0, len(idx), 256)])
    val_mel = torch.cat([fe(center_crop(val_ref["waveforms"][i:i + 256]).to(device).float()) for i in range(0, len(val_ref["labels"]), 256)])
    nn_gen = C.nearest_neighbor_distance(gen_mel.to(device), bank).cpu().numpy()
    nn_val = C.nearest_neighbor_distance(val_mel, bank).cpu().numpy()
    rows = []
    for c in range(n_classes):
        g, v = nn_gen[gen_y.numpy() == c], nn_val[val_ref["labels"].numpy() == c]
        rows.append({"class": class_names[c], "median NN validation": np.median(v), "median NN generated": np.median(g),
                     "generated / validation": np.median(g) / np.median(v),
                     "generated closer than val 1%": (g < np.percentile(v, 1)).mean()})
    memo = pd.DataFrame(rows).set_index("class")
    display(memo.round(3))
    summary[n] = {"final_loss": losses[-1] if losses else None, "memorisation": memo.round(4).reset_index().to_dict("records")}
    json.dump(summary[n], open(d / "summary.json", "w"), indent=2, default=float)

    plt.figure(figsize=(5, 3)); plt.plot([cfg.log_every * (i + 1) for i in range(len(losses))], losses)
    plt.title(f"training loss, {n} clips per class"); plt.xlabel("step"); plt.show()
    del unet, ema, bank, ds, sub, gen_mel, val_mel, fe
    import gc; gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

## How to read the memorisation table

* **generated / validation** near 1 (as for the full-data model, 0.96-0.99): generated clips sit
  no closer to the training clips than unseen real clips do, so the model is generating, not
  copying.
* Well below 1, or a large **generated closer than val 1%** fraction: the model reproduces its
  training clips. Expect this to be worse at 64 clips per class than at 256. The scarcity
  notebook then still runs, but its "synthetic" condition for that budget should be read as
  "augmented copies of the real clips", and reported that way.